In [2]:
def save_snapshot(change):
    if change['new']:
        file_path = 'snapshots/' + str(uuid.uuid1()) + '.jpg'
        
        with open(file_path, 'wb') as f:
            f.write(image.value)
            
        snapshot_image.value = image.value

# Attach snapshot behavior to right bumper (Button 5)
controller.buttons[5].observe(save_snapshot, names='value')

In [3]:
# ==============================================================================

def joystick_to_servo_angle(x):
    """
    Transforms raw joystick values (-1.0 to 1.0) directly into servo control instructions.
    - Pushing UP (negative value) returns positive 0 to 180 degrees.
    - Pushing DOWN (positive value) returns negative 0 to -180 degrees.
    """
    target_angle = int(-x * 180)
    target_angle = max(-180, min(180, target_angle))
    
    # Send directional movement update straight over the serial line
    TTLServo.servoAngleCtrl(5, target_angle, 1, 50)
    
    # Return a layout width modification to satisfy the writeable link target constraint
    return 300

# Link the Right Stick Y-Axis (axes[3]) safely.
# We map the output to the image width attribute (which is writable) instead of a read-only trait.
servo_link = traitlets.dlink((controller.axes[3], 'value'), (image, 'width'), transform=joystick_to_servo_angle)

print("\n-> All systems matched to source layout and activated successfully without TraitErrors!")


-> All systems matched to source layout and activated successfully without TraitErrors!


In [1]:
import uuid
import subprocess
import traitlets
import ipywidgets.widgets as widgets
from jetbot import Robot
from jetbot import Camera
from jetbot import bgr8_to_jpeg
from SCSCtrl import TTLServo

# ==============================================================================
# 1. INITIALIZE ROBOT & HARDWARE BOOT
# ==============================================================================
print("Initializing Jetbot hardware...")
robot = Robot()

# Force Servo 5 to a baseline 0 degrees on initialization
TTLServo.servoAngleCtrl(5, 0, 1, 100)

# Ensure snapshot directory exists
subprocess.call(['mkdir', '-p', 'snapshots'])

# ==============================================================================
# 2. CREATE AND DISPLAY CAMERA LAYOUT
# ==============================================================================
image = widgets.Image(format='jpeg', width=300, height=300)
snapshot_image = widgets.Image(format='jpeg', width=300, height=300)

display(widgets.HBox([image, snapshot_image]))

# ==============================================================================
# 3. START FRESH CAMERA INSTANCE & LINK FEED
# ==============================================================================
camera = Camera.instance(width=224, height=224, fps=21)

camera_link = traitlets.dlink((camera, 'value'), (image, 'value'), transform=bgr8_to_jpeg)

# ==============================================================================
# 4. CREATE GAMEPAD CONTROLLER INSTANCE
# ==============================================================================
# Using index 1 exactly matching your hardware choice
controller = widgets.Controller(index=0)
display(controller)




Succeeded to open the port
Succeeded to change the baudrate
Initializing Jetbot hardware...


Controller()